# 手性粗粒化语言（CQL）模拟

**作者**：李小辉 (Xiaohui Li)  
**邮箱**：danceluck@163.com  
**ORCID**：0009-0003-8233-7191  

本 Notebook 复现论文中的核心模拟：二元手性链的粗粒化、词汇量指数增长、熵与宇称指数分析，并展示《周易》六十四卦映射。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
from chiral_language import simulate_ensemble, chain_to_hexagram

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)

## 1. 参数设置

In [ ]:
MAX_LENGTH = 12
N_SAMPLES = 100_000
BLOCK_SIZE = 2
P_LEFT = 0.5
SEED = 42

## 2. 运行模拟

In [ ]:
df = simulate_ensemble(
    max_length=MAX_LENGTH,
    n_samples=N_SAMPLES,
    block_size=BLOCK_SIZE,
    p_left=P_LEFT,
    seed=SEED
)
df.head(12)

## 3. 可视化结果

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].semilogy(df['N'], df['max_possible_vocab'], 'k--', label='Theoretical $2^N$')
axes[0,0].semilogy(df['N'], df['vocab_size'], 'bo-', label=f'Observed (block={BLOCK_SIZE})')
axes[0,0].set_xlabel('Chain length N')
axes[0,0].set_ylabel('Vocabulary size')
axes[0,0].set_title('Exponential Vocabulary Growth')
axes[0,0].legend()

axes[0,1].plot(df['N'], df['entropy_bits'], 'rs-', label='H(W)')
axes[0,1].plot(df['N'], df['N'], 'g--', label='Max = N bits')
axes[0,1].set_xlabel('Chain length N')
axes[0,1].set_ylabel('Entropy (bits)')
axes[0,1].set_title('Shannon Entropy of Coarse Tokens')
axes[0,1].legend()

axes[1,0].plot(df['N'], df['mean_abs_parity'], 'm^-', label=r'Mean $|\tau|$')
axes[1,0].plot(df['N'], 1/df['N'], 'c--', label=r'$1/N$')
axes[1,0].set_xlabel('Chain length N')
axes[1,0].set_ylabel('Mean absolute parity')
axes[1,0].set_title('Parity Averaging in Long Chains')
axes[1,0].legend()

axes[1,1].bar(df['N'], df['vocab_size']/df['max_possible_vocab'], color='teal', alpha=0.7)
axes[1,1].set_xlabel('Chain length N')
axes[1,1].set_ylabel('Observed / Max vocab')
axes[1,1].set_title('Sampling Coverage')

plt.tight_layout()
plt.savefig('../figures/cql_notebook_results.png', dpi=300)
plt.show()

## 4. 《周易》六十四卦演示

In [ ]:
rng = np.random.default_rng(SEED)
print("随机生成的 8 个六爻卦象：\n")
for i in range(8):
    hex_chain = ''.join(rng.choice(['L', 'R'], size=6))
    print(f"{i+1}. {chain_to_hexagram(hex_chain)}")

## 5. 理论验证：2^N 公式

In [ ]:
for N in [1, 2, 3, 6, 10, 12]:
    print(f"N={N:2d}: 2^N = {2**N:6d}")